# 3-Stage 평가서버 호환 베이스라인 — 학습

공개 예제 각 5건으로 모델을 학습하고 실제 참가자 제출구조와 동일한 경로에 체크포인트를 저장합니다.

In [ ]:
%pip install -r requirements.txt

## 1. 라이브러리·모델 구조·학습함수

In [ ]:
from pathlib import Path
import itertools, json, os, random
import cv2, numpy as np, pandas as pd, torch
from torch import nn
from torchvision.models.video import mvit_v2_s
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_320_fpn,
)


In [ ]:
ROOT=Path.cwd(); DATA=ROOT/'data'; MODEL=ROOT/'model'
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(20260825); random.seed(20260825)

# Stage1: 실제 제출 점수(0.316)가 팀 베이스라인 MViT(0.53397)보다 낮게 나와서 원복함 -
# 핸드크래프트 피처+로지스틱회귀는 합성 재녹화에만 과적합된 것으로 보임(실제 재녹화
# 데이터로 검증 전까지는 미사용). 아래 fit_stage1()은 MViT, 실험용 핸드크래프트 버전은
# fit_stage1_experimental_handcrafted()로 따로 둠(기본 실행 대상 아님).
SIZE=224
S1_MEAN=torch.tensor([0.45,0.45,0.45])[:,None,None,None]
S1_STD=torch.tensor([0.225,0.225,0.225])[:,None,None,None]
FEATURE_DIM=5  # 실험용 핸드크래프트 피처 차원

# Stage2: 충돌/진입 시점 (COCO 사전학습 탐지기 + 학습형 재정렬기)
VEHICLE_CLASSES={'car','motorcycle','bus','truck'}
SCORE_THR=0.2  # 0.5->0.2: 358영상/16248프레임 캐시 검증(mean IoU 0.378->0.442, 히트율 45.2%->53.1%, 0.2 밑은 수확체감)
RERANK_FEATURES=['cx','cy','bw','bh','score','aspect']

# Stage3: 가감속/조향 (optical flow 휴리스틱, 학습 없음 - 임계값만 보정)
ACCEL=['ACCELERATING','DECELERATING','CONSTANT','STOPPED']
STEER=['LEFT','STRAIGHT','RIGHT']
FLOW_SIZE=(160,90)


In [ ]:
def load_frames(path):
    cap=cv2.VideoCapture(str(path)); out=[]
    while True:
        ok,bgr=cap.read()
        if not ok: break
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    if not out: raise ValueError(f'cannot decode: {path}')
    return out

def sample_frames(frames,n=8):
    idx=np.linspace(0,len(frames)-1,min(n,len(frames))).round().astype(int)
    return [frames[i] for i in idx]

# ---- Stage1(실제 사용): MViT 입력용 클립 ----
def _crop_tensor(rgb,size=224):
    h,w=rgb.shape[:2]; scale=size/min(h,w)
    nh,nw=max(size,round(h*scale)),max(size,round(w*scale))
    rgb=cv2.resize(rgb,(nw,nh),interpolation=cv2.INTER_AREA)
    y,x=(nh-size)//2,(nw-size)//2
    return torch.from_numpy(rgb[y:y+size,x:x+size].copy()).permute(2,0,1).float()/255

def _clip(path,n=16,center=None):
    frames=load_frames(path); total=len(frames)
    if center is None: idx=np.linspace(0,total-1,n).round().astype(int)
    else: idx=np.clip(center-n//2+np.arange(n),0,total-1)
    x=torch.stack([_crop_tensor(frames[int(i)]) for i in idx],1)
    return x,total

# ---- Stage1(실험용, 미사용 - fit_stage1_experimental_handcrafted 전용): 재녹화 시뮬레이션 ----
def _moire_overlay(frame,rng):
    h,w=frame.shape[:2]; freq=rng.uniform(0.15,0.4); phase=rng.uniform(0,np.pi)
    yy,xx=np.mgrid[0:h,0:w]; grid=0.5+0.5*np.sin(freq*(xx+yy)+phase)
    strength=rng.uniform(6,18)
    out=frame.astype(np.float32)+(grid[...,None]-0.5)*strength
    return np.clip(out,0,255).astype(np.uint8)

def _flicker_stack(frames,rng):
    period=rng.uniform(3.5,9.0); amp=rng.uniform(0.06,0.16)
    return [np.clip(f.astype(np.float32)*(1+amp*np.sin(2*np.pi*i/period)),0,255).astype(np.uint8)
            for i,f in enumerate(frames)]

def _double_compress(frame,rng):
    quality=rng.randint(25,55); bgr=cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)
    ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
    bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)

def _add_border(frame,rng):
    h,w=frame.shape[:2]; b=int(min(h,w)*rng.uniform(0.02,0.06)); out=frame.copy()
    out[:b]=out[-b:]=out[:,:b]=out[:,-b:]=0
    return out

def simulate_rerecording(frames,seed):
    rng=random.Random(seed)
    frames=_flicker_stack(frames,rng)
    frames=[_moire_overlay(f,rng) for f in frames]
    frames=[_double_compress(f,rng) for f in frames]
    frames=[_add_border(f,rng) for f in frames]
    return frames

def benign_augment(frames,seed):
    rng=random.Random(seed); gain=rng.uniform(0.85,1.15); quality=rng.randint(75,95); out=[]
    for f in frames:
        bright=np.clip(f.astype(np.float32)*gain,0,255).astype(np.uint8)
        bgr=cv2.cvtColor(bright,cv2.COLOR_RGB2BGR)
        ok,buf=cv2.imencode('.jpg',bgr,[cv2.IMWRITE_JPEG_QUALITY,quality])
        bgr=cv2.imdecode(buf,cv2.IMREAD_COLOR)
        out.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    return out

def _fft_high_freq_ratio(gray):
    f=np.fft.fftshift(np.fft.fft2(gray.astype(np.float32))); mag=np.abs(f)
    h,w=gray.shape; cy,cx=h//2,w//2
    yy,xx=np.mgrid[0:h,0:w]; r=np.sqrt((yy-cy)**2+(xx-cx)**2)
    radius=min(h,w)*0.15
    return float(mag[r>radius].sum()/(mag.sum()+1e-6))

def _blockiness(gray):
    g=gray.astype(np.float32); h,w=g.shape; h,w=h-h%8,w-w%8; g=g[:h,:w]
    boundary=np.abs(np.diff(g[:,7:w:8],axis=1)).mean() if w>8 else 0.0
    boundary+=np.abs(np.diff(g[7:h:8,:],axis=0)).mean() if h>8 else 0.0
    interior=np.abs(np.diff(g,axis=1)).mean()+np.abs(np.diff(g,axis=0)).mean()
    return float(boundary/(interior+1e-6))

def extract_features(frames):
    frames=sample_frames(frames,8)
    grays=[cv2.cvtColor(f,cv2.COLOR_RGB2GRAY) for f in frames]
    resized=[cv2.resize(g,(256,256)) for g in grays]
    fft_ratio=float(np.mean([_fft_high_freq_ratio(g) for g in resized]))
    brightness=np.array([g.mean() for g in grays],dtype=np.float32)
    flicker_std=float(brightness.std()/(brightness.mean()+1e-6))
    border=float(np.mean([np.concatenate([g[:4].ravel(),g[-4:].ravel(),g[:,:4].ravel(),g[:,-4:].ravel()]).mean() for g in grays]))
    blur=float(np.mean([cv2.Laplacian(g,cv2.CV_64F).var() for g in resized]))
    block=float(np.mean([_blockiness(g) for g in resized]))
    return np.array([fft_ratio,flicker_std,border,blur,block],dtype=np.float32)


In [ ]:
# ---- Stage1(실제 사용): MViTv2-S (팀 베이스라인 원안, 0.53397로 검증됨) ----
class Stage1MViT(nn.Module):
    def __init__(self):
        super().__init__(); self.net=mvit_v2_s(weights=None)
        self.net.head[1]=nn.Linear(self.net.head[1].in_features,2)
    def forward(self,x): return self.net(x)

# ---- Stage2: COCO 탐지기 + 다중객체 트랙 기반 진입 판정 ----
def load_detector():
    weights=FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
    model=fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights); model.eval()
    return model,weights.transforms(),weights.meta['categories']

@torch.inference_mode()
def detect_all_vehicles(model,transform,categories,frame,score_thr=0.2):
    x=transform(torch.from_numpy(frame).permute(2,0,1))
    out=model([x])[0]; candidates=[]
    for box,label,score in zip(out['boxes'],out['labels'],out['scores']):
        if score<score_thr or categories[label] not in VEHICLE_CLASSES: continue
        x0,y0,x1,y1=box.tolist(); candidates.append((x0,y0,x1,y1,float(score)))
    return candidates

def motion_energy(frames):
    grays=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),(320,180)) for f in frames]
    diffs=[cv2.absdiff(grays[i],grays[i-1]).mean() for i in range(1,len(grays))]
    return np.array([diffs[0]]+diffs,dtype=np.float32)

def find_collision_frame(frames):
    # 시작부만 고정 8프레임 제외, 끝은 제외 안 함: CCD(801개 자차관여 실측)로 재보정.
    # 원래(비율 기반 margin=10%, 시작+끝 둘 다 제외)는 사고 클립 특성상 문제 있었음 -
    # 충돌이 늘 뒷부분(50프레임 중 30~49, 최솟값이 30!)이라 끝 10% 제외가 늦게 발생한
    # 진짜 충돌을 통째로 못 찾게 막았음. 비율 기반 margin=0(끝 제외 없음)도 시도했지만
    # DACON 공개 5샘플 중 하나(자차 무관 배경사고 영상)에서 초반 카메라 흔들림을
    # 충돌로 오검출(MAE 2.40->6.40). 영상 길이에 비례하는 "비율"보다 "고정 프레임 수"가
    # 더 타당하다고 보고 시작 8프레임만 고정 제외 + 끝 제외 없음으로 재시도: CCD 기준
    # MAE 6.22(최선), within±3 50.9%, 공개 5샘플에서도 오검출 없음(MAE 2.40 유지).
    energy=motion_energy(frames)
    start_exclude=min(8,max(0,len(energy)-1))
    window=energy[start_exclude:]
    return int(np.argmax(window))+start_exclude

# ---- Stage2 차선 추정(entry_frame 공식 정의 - "피해차량 바퀴가 피의차량 차선에
# 최초로 닿는 시점", talkboard 417186/417277) ----
def _fit_lane_side(points):
    if len(points)<2: return None
    ys=np.array([p[1] for p in points],dtype=np.float64); xs=np.array([p[0] for p in points],dtype=np.float64)
    if ys.std()<1e-3: return None
    m,b=np.polyfit(ys,xs,1); return float(m),float(b)

def detect_lane_boundaries(frame):
    # 사다리꼴 ROI(화면 하단 30%, 원근상 이보다 멀면 차선이 거의 수평이 돼 구분 불가)
    # 로 도로 밖 직선을 배제하고 Canny+HoughLinesP, 좌/우 기울기 부호로 분리해 피팅.
    h,w=frame.shape[:2]; roi_top=int(h*0.7)
    mask=np.zeros((h,w),dtype=np.uint8)
    trapezoid=np.array([[0,h],[int(w*0.3),roi_top],[int(w*0.7),roi_top],[w,h]])
    cv2.fillPoly(mask,[trapezoid],255)
    gray=cv2.cvtColor(frame,cv2.COLOR_RGB2GRAY)
    edges=cv2.bitwise_and(cv2.Canny(gray,50,150),mask)
    lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=25,minLineLength=int(h*0.08),maxLineGap=30)
    if lines is None: return None
    left_pts,right_pts=[],[]
    for x1,y1,x2,y2 in lines.reshape(-1,4):
        if x2==x1: continue
        slope=(y2-y1)/(x2-x1)
        if abs(slope)<0.4: continue
        (left_pts if slope<0 else right_pts).extend([(x1,y1),(x2,y2)])
    left,right=_fit_lane_side(left_pts),_fit_lane_side(right_pts)
    if left is None or right is None: return None
    return left,right

def _default_lane(h,w):
    # 검출 실패시 기본값(DACON 공개 5샘플 전부 실패 - talkboard 417288 참고) - 자차가
    # 차선 중앙, 차선폭은 화면폭의 38%로 가정.
    half=0.19*w
    left=tuple(np.polyfit([h,0],[w/2-half,w/2],1)); right=tuple(np.polyfit([h,0],[w/2+half,w/2],1))
    return left,right

def estimate_ego_lane(frames,ts,h,w):
    ts=list(ts); lefts,rights=[],[]
    for t in ts:
        fit=detect_lane_boundaries(frames[t])
        if fit is None: continue
        lefts.append(fit[0]); rights.append(fit[1])
    if len(lefts)<max(2,len(ts)//4): return _default_lane(h,w)
    return tuple(np.median(lefts,axis=0)),tuple(np.median(rights,axis=0))

# ---- Stage2 다중객체 트랙 기반 진입 판정(정민 v10 이식) ----
# 2026-09-13: 팀원(정민)이 CCD 실측 entry_frame/entry_side/evasion_space 정답 36개(사람이
# 직접 검수, review_status=DONE)를 공유해줘서 entry_frame이 이번 세션 처음으로 정량
# 검증 가능해졌다(external/jungmin_labels/legacy_labels.csv). 기존 방식(프레임 독립
# 재정렬+"조금이라도 겹치면 진입")은 Accuracy@0.3s=16.7%(6/36), MAE=1.79초 - 원인
# 진단 결과 탐지 실패가 아니라(0%) "차량이 처음부터 차선 근처에 있으면 프레임 0
# 근처에서 바로 진입으로 오판"이 대부분(정민 코드 주석: "v9의 지배적 초반 오탐 원인").
#
# 정민의 tools/stage2_track_v10_core.py(다중객체 그리디 트래킹 + "이전엔 코리도 밖에
# 있다가 이후 안으로 전환하는 순간"만 진입으로 인정)를 이식해 같은 36개로 우리
# 탐지기로 직접 재검증 - Accuracy@0.3s=22.2%(8/36), MAE=1.11초, entry_side=75.0%,
# evasion_space=52.8%로 전부 기존보다 나음(이 36개로 우리가 설계한 적이 없어 사실상
# 독립 재현). 프레임별 독립 재정렬(reranker.pt) 대신 이 트래킹으로 entry_frame/
# entry_side/evasion_space를 전부 교체.
def _lane_bounds(lane,y):
    values=[float(m)*float(y)+float(b) for m,b in lane]
    return min(values),max(values)

def _track_geometry(box,width,height):
    x0,y0,x1,y1,score=map(float,box[:5])
    return ((x0+x1)/(2*width),(y0+y1)/(2*height),max(1.0,x1-x0)/width,max(1.0,y1-y0)/height,float(score))

def _link_cost(previous,current,width,height,gap=1):
    ax,ay,aw,ah,_=_track_geometry(previous,width,height)
    bx,by,bw,bh,_=_track_geometry(current,width,height)
    distance=np.hypot(ax-bx,ay-by)/max(0.035,0.5*(aw+bw),0.5*(ah+bh))
    scale=abs(np.log((bw*bh+1e-6)/(aw*ah+1e-6)))
    return float(distance/max(1.0,gap)+0.35*scale+0.08*(gap-1))

def _build_tracks(all_boxes,width,height,max_gap=3,max_cost=2.2):
    tracks=[]
    for time_index in sorted(all_boxes):
        boxes=sorted((tuple(map(float,box[:5])) for box in all_boxes[time_index]),
                     key=lambda box:(-box[4],box[0],box[1]))[:16]
        candidates=[]
        for track_index,track in enumerate(tracks):
            gap=time_index-track['times'][-1]
            if 1<=gap<=max_gap:
                for box_index,box in enumerate(boxes):
                    cost=_link_cost(track['boxes'][-1],box,width,height,gap)
                    if cost<=max_cost: candidates.append((cost,track_index,box_index))
        used_tracks,used_boxes=set(),set()
        for _,track_index,box_index in sorted(candidates):
            if track_index in used_tracks or box_index in used_boxes: continue
            tracks[track_index]['times'].append(int(time_index)); tracks[track_index]['boxes'].append(boxes[box_index])
            used_tracks.add(track_index); used_boxes.add(box_index)
        for box_index,box in enumerate(boxes):
            if box_index not in used_boxes: tracks.append({'times':[int(time_index)],'boxes':[box]})
    return tracks

def _track_inside_ratio(box,lane):
    x0,_,x1,y1=map(float,box[:4]); left,right=_lane_bounds(lane,y1)
    overlap=max(0.0,min(x1,right)-max(x0,left))
    return overlap/max(1.0,x1-x0)

def _track_features(track,lane,collision,width,height):
    times,boxes=track['times'],track['boxes']
    usable=[(t,b) for t,b in zip(times,boxes) if t<=collision+3]
    if not usable or not any(t<=collision for t,_ in usable):
        return {'score':-1e9,'entry':None,'side':'RIGHT','crossing':False,'terminal_distance':10**9,'length':0,'continuity':0.0}
    times=[x[0] for x in usable]; boxes=[x[1] for x in usable]
    inside=[_track_inside_ratio(box,lane) for box in boxes]
    crossing_index=None
    for index in range(len(times)):
        future=inside[index:min(len(inside),index+3)]
        previous=inside[max(0,index-2):index]
        if inside[index]>=0.10 and sum(value>=0.10 for value in future)>=min(2,len(future)):
            # 처음부터 코리도 안에 있는 건 진입 증거가 아니다 - 직전(median)이 밖(<10%)
            # 이었다가 지금 안으로 전환되는 순간만 인정(초반 오탐 방지).
            if previous and float(np.median(previous))<0.10:
                crossing_index=index; break
    entry=times[crossing_index] if crossing_index is not None else None
    side_samples=boxes[max(0,(crossing_index or 0)-3):(crossing_index or 0)+1]
    offsets=[]
    for box in side_samples:
        x0,_,x1,y1=box[:4]; left,right=_lane_bounds(lane,y1)
        offsets.append((x0+x1)/2-(left+right)/2)
    side='LEFT' if offsets and float(np.median(offsets))<0 else 'RIGHT'
    terminal_index=int(np.argmin([abs(t-collision) for t in times]))
    terminal_time,terminal=times[terminal_index],boxes[terminal_index]
    x0,y0,x1,y1,confidence=terminal
    area=(x1-x0)*(y1-y0)/max(1.0,width*height)
    terminal_distance=abs(terminal_time-collision)
    continuity=len(times)/max(1,times[-1]-times[0]+1)
    areas=[(b[2]-b[0])*(b[3]-b[1]) for b in boxes]
    expansion=np.log((areas[-1]+1)/(areas[0]+1))/max(1,len(areas)-1)
    lateral=0.0
    if len(boxes)>=2:
        lateral=abs(((boxes[-1][0]+boxes[-1][2])-(boxes[0][0]+boxes[0][2]))/(2*width))
    score=(2.0*_track_inside_ratio(terminal,lane)+1.2*min(1.0,area/0.08)+0.45*confidence
           +0.65*continuity+0.35*min(1.0,max(0.0,expansion)/0.08)+0.45*min(1.0,lateral/0.15)-0.18*terminal_distance)
    if crossing_index is not None and entry<=collision:
        score+=0.9+0.25*min(1.0,(collision-entry)/max(1,collision))
    return {'score':float(score),'entry':entry,'side':side,'crossing':crossing_index is not None,'terminal':terminal,
            'terminal_distance':terminal_distance,'length':len(times),'continuity':float(continuity)}

def trajectory_scene(all_boxes,lane,height,width,collision):
    tracks=_build_tracks(all_boxes,width,height)
    featured=[(track,_track_features(track,lane,collision,width,height)) for track in tracks]
    eligible=[(track,feature) for track,feature in featured if feature['terminal_distance']<=5 and feature['length']>=2]
    selected=max(eligible,key=lambda item:item[1]['score'],default=None)
    if selected is None:
        return {'entry':int(max(0,collision)),'side':'RIGHT','evasion':0}
    track,feature=selected
    if feature['entry'] is None:
        pre_collision=[t for t in track['times'] if t<=collision]
        entry=min(pre_collision) if pre_collision else max(0,collision)
    else:
        entry=int(feature['entry'])
    terminal=feature['terminal']
    x0,_,x1,y1=terminal[:4]; left,right=_lane_bounds(lane,y1)
    lane_width=max(1.0,right-left)
    evasion=int(max(max(0.0,x0-left),max(0.0,right-x1))>=0.32*lane_width)
    return {'entry':min(int(collision),int(entry)),'side':feature['side'],'evasion':evasion}

def find_entry_and_scene(model,transform,categories,frames,collision_frame,reranker=None):
    h,w=frames[0].shape[:2]
    high=min(len(frames)-1,collision_frame+3)
    sampled=sorted(set(np.rint(np.linspace(0,high,min(96,high+1))).astype(int).tolist()))
    all_boxes={}
    for t in sampled:
        boxes=detect_all_vehicles(model,transform,categories,frames[t],score_thr=SCORE_THR)
        if boxes: all_boxes[t]=boxes
    lane=estimate_ego_lane(frames,sampled,h,w) if sampled else _default_lane(h,w)
    result=trajectory_scene(all_boxes,lane,h,w,collision_frame)
    return result['entry'],result['side'],result['evasion']

# ---- Stage3: optical flow (학습 없음, 임계값만 보정) ----
def compute_flow_series(frames):
    small=[cv2.resize(cv2.cvtColor(f,cv2.COLOR_RGB2GRAY),FLOW_SIZE) for f in frames]
    n=len(small)//2; w,h=FLOW_SIZE
    road=slice(int(h*0.55),h); horizon=slice(int(h*0.25),int(h*0.55))
    speed=np.zeros(n,dtype=np.float32); steer=np.zeros(n,dtype=np.float32)
    for t in range(n):
        i0=min(2*t,len(small)-3); i1=i0+2
        flow=cv2.calcOpticalFlowFarneback(small[i0],small[i1],None,0.5,2,15,3,5,1.2,0)
        mag=np.sqrt(flow[...,0]**2+flow[...,1]**2)
        speed[t]=float(np.median(mag[road])); steer[t]=float(np.median(flow[horizon,:,0]))
    return speed,steer

def _smooth(x,k=3):
    if len(x)<2*k+1: return x
    width=2*k+1; padded=np.pad(x,(k,k),mode='edge')
    return np.median(np.lib.stride_tricks.sliding_window_view(padded,width),axis=-1)

def classify(speed,steer,stopped_thr,accel_eps,steer_thr):
    # steer 스무딩 추가했다가(LOVO 0.670->0.710) 실제 점수가 0.521->0.48->0.47로
    # 계속 떨어져서 원복. Macro-F1 공식(팀 이슈 #3: 0.7*accel+0.3*steer, STOPPED 제외)
    # 으로 다시 그리드서치해도 스무딩 여부와 무관하게 같은 임계값이 최적이라, 스무딩
    # 자체의 문제라기보다 "공개 5비디오 로컬검증이 실제 숨은 평가셋과 거의 무관하다"는
    # 구조적 한계로 보임(팀장님도 반대방향 동일 현상 - 이슈 #10). 실측 검증된 상태로 복귀.
    # 2026-09-13: 스무딩을 평균->중앙값으로 교체(팀원 정민 submit-3 참고, 공개 50라벨 acc
    # 변화 없이 회귀 없음 확인).
    speed_s=_smooth(speed); n=len(speed_s); accel_out,steer_out=[],[]
    for t in range(n):
        if speed_s[t]<stopped_thr: accel_out.append('STOPPED')
        else:
            lo,hi=max(0,t-3),min(n,t+4)
            width=(hi-1)-lo
            slope=(speed_s[hi-1]-speed_s[lo])*(6/width) if width>0 else 0.0
            accel_out.append('ACCELERATING' if slope>accel_eps else 'DECELERATING' if slope<-accel_eps else 'CONSTANT')
        s=steer[t]
        # 좌회전->배경이 화면에서 오른쪽으로 흐름(flow_x 양수). AIHub 실측 자이로+실제 프레임으로 검증된 부호.
        steer_out.append('LEFT' if s>steer_thr else 'RIGHT' if s<-steer_thr else 'STRAIGHT')
    return accel_out,steer_out

In [ ]:
def _video_bpp(path):
    """size(bytes)*8 / (frame_count*width*height) - 해상도/길이 무관 압축난이도 지표.
    공개 10샘플에서 원본 vs 재녹화가 깨끗하게 갈림(LOO 9~10/10) - 단, DACON이
    재녹화 예제는 "실제 재촬영 아닌 파생 예제"라 명시해서 실제 평가셋에 안 통할
    위험 있음(인코더/코덱 차이가 DACON 예제 생성 파이프라인 특성일 수 있음).
    그래도 MViT(LOO 0/10, 페어 암기로 역방향)보다 훨씬 강한 로컬 신호라 공격적으로
    채택 - MViT를 보조(가중치 0.25)로만 섞어서 한쪽에만 전부 걸지는 않음."""
    cap=cv2.VideoCapture(str(path))
    nframes=cap.get(cv2.CAP_PROP_FRAME_COUNT); w=cap.get(cv2.CAP_PROP_FRAME_WIDTH); h=cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    if nframes<=0 or w<=0 or h<=0: return 0.0
    return Path(path).stat().st_size*8/(nframes*w*h)

def fit_stage1():
    """팀 베이스라인 원안(MViTv2-S, ImageNet 가중치 없이 5+5 공개샘플로 1 epoch 학습)
    + bpp(비트레이트) 신호, 가중치 0.75(원복).
    LOO 스윕(stage1_blend_sweep.py)은 weight=1.0(순수 bpp)이 9/10으로 0.75(8/10)보다
    낫다고 나와서 한 번 1.0으로 올렸으나, 실제 제출 결과 0.589(0.75) -> 0.5783(1.0)로
    오히려 하락 - 로컬 LOO(10샘플)가 이번에도 실제 방향과 어긋남(공개 소량 검증의
    반복되는 한계). 실측 A/B(0.75 real 0.589 > 1.0 real 0.5783)를 우선해서 0.75로
    되돌림. MViT 학습/추론 코드는 그대로 둠."""
    out=MODEL/'stage1'; out.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(DATA/'stage1/labels.csv')
    model=Stage1MViT().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),1e-4)
    model.train()
    for r in df.sample(frac=1,random_state=20260825).itertuples():
        x,_=_clip(DATA/'stage1'/r.path,16); x=(x-S1_MEAN)/S1_STD
        y=torch.tensor([0 if r.label=='ORIGINAL' else 1],device=DEVICE)
        loss=nn.functional.cross_entropy(model(x[None].to(DEVICE)),y)
        opt.zero_grad(); loss.backward(); opt.step()

    bpps={r.ID:_video_bpp(DATA/'stage1'/r.path) for r in df.itertuples()}
    o_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='ORIGINAL']
    r_vals=[bpps[r.ID] for r in df.itertuples() if r.label=='RERECORDED']
    bpp_thr=(max(o_vals)+min(r_vals))/2; bpp_scale=max((min(r_vals)-max(o_vals))/2,1e-6)
    print(f'  stage1 bpp_thr={bpp_thr:.5f} bpp_scale={bpp_scale:.5f} (원본 bpp<={max(o_vals):.4f}, 재녹화 bpp>={min(r_vals):.4f})')

    torch.save({'model':model.net.state_dict(),'size':224,'frames':16,
                'bpp_thr':bpp_thr,'bpp_scale':bpp_scale,'bpp_weight':0.75}, out/'best.pt')

def fit_stage1_experimental_handcrafted():
    """실험용(기본 실행 대상 아님) - 합성 재녹화+benign 증강으로 만든 데이터로 5개 핸드크래프트
    피처+로지스틱회귀 학습. 로컬 합성데이터 정확도는 95~98%였지만 실제 제출 0.316으로
    베이스라인(0.53397)보다 나빴다 - 합성 artefact 과적합으로 추정. 실제 재녹화 데이터
    확보 전까지는 model/stage1/best.pt를 덮어쓰지 않도록 별도 경로에 저장한다."""
    out=MODEL/'stage1_experimental'; out.mkdir(parents=True,exist_ok=True)
    originals=sorted((DATA/'stage1/original').glob('*.mp4'))+sorted((DATA/'stage2/videos').glob('*.mp4'))+sorted((DATA/'stage3/videos').glob('*.mp4'))
    rerecorded=sorted((DATA/'stage1/rerecorded').glob('*.mp4'))
    X,y=[],[]
    for path in originals:
        frames=sample_frames(load_frames(path),8)
        X.append(extract_features(frames)); y.append(0)
        for v in range(3):
            X.append(extract_features(benign_augment(frames,seed=hash((path.name,v))&0xFFFF))); y.append(0)
            X.append(extract_features(simulate_rerecording(frames,seed=hash((path.name,v,'r'))&0xFFFF))); y.append(1)
    for path in rerecorded:
        X.append(extract_features(load_frames(path))); y.append(1)
    X,y=np.stack(X),np.array(y,dtype=np.float32)

    mean,std=X.mean(0),X.std(0)+1e-6
    Xn=torch.tensor((X-mean)/std,dtype=torch.float32); yt=torch.tensor(y)
    weight=torch.zeros(X.shape[1],requires_grad=True); bias=torch.zeros(1,requires_grad=True)
    opt=torch.optim.Adam([weight,bias],lr=0.1)
    for _ in range(300):
        loss=nn.functional.binary_cross_entropy_with_logits(Xn@weight+bias,yt)
        opt.zero_grad(); loss.backward(); opt.step()
    acc=float(((Xn@weight+bias>0).float()==yt).float().mean())
    print(f'  stage1(실험용) train acc: {acc:.3f} (합성 데이터 기준 - 실제 0.316으로 검증됨, 참고용)')
    torch.save({'weight':weight.detach(),'bias':bias.detach(),'feat_mean':mean,'feat_std':std}, out/'best.pt')

def fit_stage2():
    """COCO 사전학습 탐지기는 그대로 저장(파인튜닝 아님). 재정렬기(reranker.pt)는 AIHub
    실라벨(597, 차대차 카테고리)로 별도 학습한 산출물 - 공개 5샘플엔 그런 라벨이 없어서
    여기서 재현 불가. 이미 model/stage2/reranker.pt가 있으면(로컬에서 미리 학습) 그대로 두고,
    없으면 score*area 폴백으로 동작 (detect_vehicles 참고)."""
    out=MODEL/'stage2'; out.mkdir(parents=True,exist_ok=True)
    model,_,_=load_detector()
    torch.save(model.state_dict(), out/'detector.pth')
    reranker_path=out/'reranker.pt'
    print(f"  reranker.pt {'존재 - 유지' if reranker_path.exists() else '없음 - score*area 폴백으로 동작'}")

def fit_stage3():
    """stopped_thr/accel_eps: AIHub 실측 CAN(10영상/35982프레임, LOVO 0.687) 기준 -
    실제 제출로 0.521->0.53113 개선 확인됨(real validated).
    steer_thr: SullyChen driving_dataset(45406프레임, 실측 조향각 degree) 기준으로 교체
    (2026-09-12) - AIHub CAN의 steering_angle 필드가 전 영상·전 차종에서 100% 0으로
    죽어있어서(placeholder) 각속도로 대체했었는데, 진짜 조향각 데이터로 재보정한 것.
    optical-flow steer_proxy와의 상관계수 -0.845(AIHub 각속도 -0.519보다 강함). steer_thr
    그리드서치: 이전 0.464는 정확도 0.865, 0.65가 0.886으로 더 나음 - 아직 실제 제출로는
    미검증(다음 제출 대상). 이전 값들은 stage3_can_candidate.PREV_CAN_CANDIDATE 참고."""
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    params={'stopped_thr':0.1,'accel_eps':0.01,'steer_thr':0.65}  # steer_thr만 SullyChen으로 재보정
    print(f'  stage3 보정값 적용: {params} (steer_thr는 SullyChen 45406프레임 검증, acc 0.886)')
    torch.save(params, out/'best.pt')

def fit_stage3_grid_search_50samples():
    """원래 로직(참고/원복용, 기본 실행 대상 아님) - 공개 라벨 50개(6초 간격) 기준 그리드서치.
    steer 스무딩 추가가 로컬(LOVO)엔 나았지만(0.670->0.710) 실제 제출은 0.521->0.48로
    떨어져서 되돌린 이력 있음(그리드도 6x6x6가 10x10x10보다 실측 안전). 결과값
    (stopped_thr=0.1, accel_eps=0.3, steer_thr=0.28)이 real score 0.521로 검증된 상태
    (CAN 기반 fit_stage3()의 0.53113보다 낮음 - 이제는 CAN 기반이 기본)."""
    out=MODEL/'stage3'; out.mkdir(parents=True,exist_ok=True)
    labels=pd.read_csv(DATA/'stage3/labels.csv')
    cache={}
    for vid_id,group in labels.groupby('ID'):
        frames=load_frames(DATA/'stage3/videos'/f'{vid_id}.mp4')
        cache[vid_id]=(compute_flow_series(frames),group)

    best=None
    grid=itertools.product(np.linspace(0.1,1.5,6), np.linspace(0.02,0.3,6), np.linspace(0.1,1.0,6))
    for stopped_thr,accel_eps,steer_thr in grid:
        correct=total=0
        for vid_id,((speed,steer),group) in cache.items():
            accel_pred,steer_pred=classify(speed,steer,stopped_thr,accel_eps,steer_thr)
            for row in group.itertuples():
                idx=min(row.sample_index,len(accel_pred)-1); total+=2
                correct+=accel_pred[idx]==row.accel_label; correct+=steer_pred[idx]==row.steer_label
        acc=correct/total
        if best is None or acc>best[0]: best=(acc,stopped_thr,accel_eps,steer_thr)
    acc,stopped_thr,accel_eps,steer_thr=best
    print(f'  stage3 calibrated acc: {acc:.3f} (stopped_thr={stopped_thr:.3f}, accel_eps={accel_eps:.3f}, steer_thr={steer_thr:.3f})')
    torch.save({'stopped_thr':stopped_thr,'accel_eps':accel_eps,'steer_thr':steer_thr}, out/'best.pt')

## 2. Stage 1·2·3 학습

In [ ]:
print('device:',DEVICE)
fit_stage1(); print('Stage 1 완료')
fit_stage2(); print('Stage 2 완료')
fit_stage3(); print('Stage 3 완료')

In [ ]:
for p in sorted((ROOT/'model').rglob('*')):
    if p.is_file(): print(p.relative_to(ROOT),f'{p.stat().st_size/1024**2:.1f} MB')